# **Descrição do algoritmo**




In [ ]:
#Imports gerais
import numpy as np
import cv2
import sys
from PIL import Image, ImageDraw, ImageFont


#Imports do colab
from google.colab.patches import cv2_imshow
from google.colab import files



In [ ]:
# Liga os blocos de Debug. Serve para visualização de imagens intermediárias e máscaras.
# Aumenta significativamente o tempo de execução do código já que ele imprime imagens com resolução alta várias vezes.
# Não é necessário para o funcionamento do código, apenas para visualização do que está acontecendo internamente.

Debug = False

In [ ]:
def formatRes(r):
    omega = "\u03A9"

    if r < 1e3:
        return f'{r:.0f} {omega}'
    elif r < 1e6:
        return f'{r/1e3:.1f} k{omega}'
    elif r < 1e9:
        return f'{r/1e6:.1f} M{omega}'
    else:
        return f'{r/1e9:.1f} G{omega}'

In [ ]:
def colorDec(cor,canBlack = True):
# Verificar centros usando resistores.

    centroCores = {
        'Vermelho':[0,179],
        'Laranja': 10,
        'Amarelo':25,
        'Verde':50,
        'Azul':110,
        'Violeta':130
    }

    melhor_cor = None
    menor_dist = float('inf')
    hcor = cor[0]
    for nome, centro in centroCores.items():

        if nome == 'Vermelho':
          dist1 = np.linalg.norm(np.array(hcor) - np.array(centro[0]))
          dist2 = np.linalg.norm(np.array(hcor) - np.array(centro[1]))
          dist = min(dist1,dist2)

        else:
          dist = np.linalg.norm(
              np.array(hcor) - np.array(centro)
          )

        if dist < menor_dist:
            menor_dist = dist
            melhor_cor = nome

    if melhor_cor == 'Vermelho' or melhor_cor == 'Laranja':
            if (cor[1] < 70 and (30 < cor[2] < 60)) or (cor[1] < 20 and (40 < cor[2] < 100)):
                if cor[2] < 30 and canBlack:
                    melhor_cor = 'Preto'

                else:
                    melhor_cor = 'Cinza'

            elif cor[2] < 70:
                if cor[2] < 15 and canBlack:
                    melhor_cor = 'Preto'

                else:
                    melhor_cor = 'Marrom'

            elif cor[2] > 100:
                melhor_cor = 'Laranja'

    else:
            if (cor[1] < 70 and cor[2] < 60) or (cor[1] < 20 and cor[2] < 100):
                if cor[2] < 30 and canBlack:
                    melhor_cor = 'Preto'

                else:
                    melhor_cor = 'Cinza'

            elif cor[2] < 20 and canBlack:
                melhor_cor = 'Preto'

    print(melhor_cor, menor_dist)
    return melhor_cor

In [ ]:
def resistencia(cores):

  colorcode = {'Preto':0,
               'Marrom':1,
               'Vermelho':2,
               'Laranja':3,
               'Amarelo':4,
               'Verde':5,
               'Azul':6,
               'Violeta':7,
               'Cinza':8,
               }
  holder = []

  for cor in cores:
    holder.append(colorcode[cor])


  resist = (holder[0]*10 + holder[1])*10**(holder[2])
  resist = formatRes(resist)
  return resist
  print(f'Seu resistor é de {resist} omhs! ')


# **Segmentação do Resistor**


In [ ]:


# res_num = input('Qual número do res: ')
# img = cv2.imread(f'/content/drive/MyDrive/Projeto - Resistores/Testes/IMG_16{res_num}.png')
# img = cv2.imread(f'/content/drive/MyDrive/Projeto - Resistores/Fernando/Resistor_{res_num}.png')
# img = cv2.imread(f'/content/drive/MyDrive/Projeto - Resistores/Resistores/Resistor_{res_num}.png')

uploadImg = files.upload() # Permite subir imaens do seu computador
imgFile = next(iter(uploadImg))
img = cv2.imread(imgFile)


hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
h,s,v = cv2.split(hsv)

v = cv2.GaussianBlur(v,(5,5),0)
bg = cv2.GaussianBlur(v,(101,101),0)
v2 = cv2.divide(v,bg,scale=255)

clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
v2 = clahe.apply(v2)
final = cv2.merge((h,s,v2)) #Imagem com fundo muito claro para limiarização.

In [ ]:
if Debug == True:
  cv2_imshow(cv2.resize(img,(600,800))) #Imagem original (tamanho alterado para visualização)
  finalDebug = cv2.cvtColor(final, cv2.COLOR_HSV2BGR)
  cv2_imshow(cv2.resize(finalDebug,(600,800))) #Imagem clareada (tamanho alterado para visualização)

In [ ]:
                #[H, S, V]
lower = np.array([0, 0, 170])
upper = np.array([180, 45, 255])
background = cv2.inRange(final, lower, upper)

# Inverte
mask = cv2.bitwise_not(background)

kernel = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE,
    (5,5)
)

mask = cv2.morphologyEx(
    mask,
    cv2.MORPH_CLOSE,
    kernel
)



In [ ]:
if Debug == True:
  cv2_imshow(mask) #Máscara com ruído

In [ ]:

contornos, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contorno_escolhido = max(contornos, key=cv2.contourArea)
mask = np.zeros(final.shape[:2], dtype=np.uint8)
cv2.drawContours(mask, [contorno_escolhido], -1, 255, -1);


In [ ]:
if Debug == True:
  cv2_imshow(mask) #Máscara somente com o resistor

In [ ]:

radius = cv2.distanceTransform(mask, cv2.DIST_L2, 5).max()

L =int( radius if radius%2 > 0 else radius+ 1 )
kernel = cv2.getStructuringElement(
    cv2.MORPH_RECT,
    (L, L)
)


opening = cv2.morphologyEx(
    mask,
    cv2.MORPH_OPEN,
    kernel)

resistor = opening


In [ ]:
if Debug == True:
  cv2_imshow(resistor) #Máscara com o resistor sem as pernas

In [ ]:
resultado = cv2.bitwise_and(img, img, mask=resistor)

ContRes, _ = cv2.findContours(resistor, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
rect = cv2.minAreaRect(max(ContRes, key=cv2.contourArea))

(cx_rect, cy_rect), (w, h), angle = rect


if w < h:
    angle = angle + 90

M = cv2.getRotationMatrix2D((cx_rect, cy_rect), angle, 1.0)

imgRotac = cv2.warpAffine(
    resultado,
    M,
    (resultado.shape[1], resultado.shape[0]),
    flags=cv2.INTER_LINEAR,
    borderMode=cv2.BORDER_CONSTANT,
    borderValue=(0, 0, 0)
)

imgRotac2 = cv2.warpAffine(
    img,
    M,
    (img.shape[1], img.shape[0]),
    flags=cv2.INTER_LINEAR,
    borderMode=cv2.BORDER_CONSTANT,
    borderValue=(0, 0, 0)
)

resistor = cv2.warpAffine(
    resistor,
    M,
    (resistor.shape[1], resistor.shape[0]),
    flags=cv2.INTER_NEAREST,
    borderMode=cv2.BORDER_CONSTANT,
    borderValue=0
)


In [ ]:
if Debug == True:
  cv2_imshow(cv2.resize(imgRotac, (600,800)))  #Imagem após aplicação da máscara rotacionada
  cv2_imshow(cv2.resize(imgRotac2, (600,800))) #Imagem original rotacionada
  cv2_imshow(cv2.resize(resistor, (600,800)))  #Máscara rotacionada

# **Segmentação das faixas**

In [ ]:

blur = cv2.GaussianBlur(imgRotac2, (11,11), 0)
blur = cv2.bitwise_and(blur,blur,mask = resistor)

corpohsv = cv2.cvtColor(blur, cv2.COLOR_BGR2HSV)
hc,sc,vc = cv2.split(corpohsv)

LFaixas =int( L*0.2 if radius%2 > 0 else L*0.2+ 1 )
kernelFaixas = cv2.getStructuringElement(
    cv2.MORPH_RECT,
    (LFaixas, LFaixas)
)

In [ ]:
hcc = hc[resistor>0]

histh = cv2.calcHist([hcc],[0],None, [179],[0,179])
maxh = np.argmax(histh)

maskhInv = cv2.inRange(hc, int(maxh - 12), int(maxh + 12)) #Máscara do corpo

maskh = cv2.bitwise_and(
    cv2.bitwise_not(maskhInv),
    resistor
) #Máscara do que não é corpo.

maskh = cv2.morphologyEx(
    maskh,
    cv2.MORPH_OPEN,
    kernelFaixas)

In [ ]:
if Debug == True:
  cv2_imshow(maskh) #Máscara obtida a partir da filtragem da cor com valor H mais proeminente

In [ ]:
sb = cv2.bitwise_and(sc,sc,mask = maskhInv)

#Extração de cores vivas que foram filtrada como corpo.

_,masks = cv2.threshold(sb,150,255,cv2.THRESH_BINARY)

masks = cv2.morphologyEx(
    masks,
    cv2.MORPH_OPEN,
    kernelFaixas)

maskf = cv2.bitwise_or(maskh,masks)

In [ ]:
if Debug == True:
  cv2_imshow(masks) #Máscara obtida a partir da filtragem do valor de saturação (S)
  cv2_imshow(maskf) #Soma das duas máscaras obtidas

In [ ]:
gray = cv2.cvtColor(blur,cv2.COLOR_BGR2GRAY)
gray = cv2.bitwise_and(gray,gray,mask = maskhInv)

#Extração de cores escuras que foram filtradas como corpo.

_,maskgInv = cv2.threshold(gray,50,255,cv2.THRESH_BINARY)

maskg = cv2.bitwise_and(
    cv2.bitwise_not(maskgInv),
    resistor
)

maskg = cv2.morphologyEx(
    maskg,
    cv2.MORPH_OPEN,
    kernelFaixas)

maskff = cv2.bitwise_or(maskf,maskg)

In [ ]:
if Debug == True:
  cv2_imshow(maskg)  #Máscara obtida a partir da filtragem de cores escuras
  cv2_imshow(maskff) #Soma de todas as máscaras obtidas até o momento.

# **Tratamento dos contornos das faixas**


In [ ]:
contornosF, _ = cv2.findContours(maskff, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

In [ ]:
if Debug == True:
  imgRotacDebug = imgRotac.copy()
  cv2.drawContours(imgRotacDebug, contornosF, -1, (0,255,0),2);
  cv2_imshow(imgRotacDebug)

In [ ]:
#Remove contornos na mesma posição X

dados = []

for contorno in contornosF:
    M = cv2.moments(contorno)
    if M["m00"] == 0:
        continue
    cx = M["m10"] / M["m00"]
    area = cv2.contourArea(contorno)
    dados.append({
        "contorno": contorno,
        "cx": cx,
        "area": area
    })


remover = set()
for i in range(len(dados)):
    for j in range(i + 1, len(dados)):
        if abs(dados[i]["cx"] - dados[j]["cx"]) <= 5:
            # Remove o de menor área
            if dados[i]["area"] < dados[j]["area"]:
                remover.add(i)
            else:
                remover.add(j)


faixas = [
    dados[i]["contorno"]
    for i in range(len(dados))
    if i not in remover
]

In [ ]:
if Debug == True:
  imgRotacDebug = imgRotac.copy()
  cv2.drawContours(imgRotacDebug, faixas, -1, (0,255,0),2);
  cv2_imshow(imgRotacDebug)

In [ ]:
if len(faixas) >= 3:
  faixas = sorted(
      faixas,
      key=cv2.contourArea,
      reverse=True
  )[:3]

else:
  print('Menos de 3 faixas identificadas. Por favor, tire outra foto.')
  sys.exit()

In [ ]:
if Debug == True:
  imgRotacDebug = imgRotac.copy()
  cv2.drawContours(imgRotacDebug, faixas, -1, (0,255,0),2);
  cv2_imshow(imgRotacDebug)

# **Ordenação das faixas para leitura**


In [ ]:

# Ordena as Faixas para ler o resistor na ordem correta
ContRes, _ = cv2.findContours(resistor, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

x_res, y_res, w_res, h_res = cv2.boundingRect(
    max(ContRes, key=cv2.contourArea)
)

xs = []

# Centroide de cada faixa
for faixa in faixas:
    M = cv2.moments(faixa)

    px = int(M["m10"] / M["m00"])
    xs.append(px)

# Distância até a borda esquerda e direita do resistor
d_esq = [x - x_res for x in xs]
d_dir = [x_res + w_res - x for x in xs]

# Faixas mais à esquerda e mais à direita
argmin = np.argmin(xs)
argmax = np.argmax(xs)

# Faixa do meio
meio = ({0, 1, 2} - {argmin, argmax}).pop()

FdArg1 = [d_esq[argmin], d_dir[argmin]]
FdArg2 = [d_esq[argmax], d_dir[argmax]]

# Quem estiver mais próximo de uma extremidade é a primeira faixa
if min(FdArg1) < min(FdArg2):
    ordem = [argmin, meio, argmax]
else:
    ordem = [argmax, meio, argmin]

FaixasOrdenadas = [faixas[i] for i in ordem]

In [ ]:
if Debug == True:
  imgRotacDebug  = imgRotac.copy()
  imgRotacDebug2 = imgRotac.copy()

  # Retângulo do resistor
  cv2.rectangle(
      imgRotacDebug,
      (x_res, y_res),
      (x_res + w_res, y_res + h_res),
      (0, 255, 0),
      2
  )
  cv2.rectangle(
      imgRotacDebug2,
      (x_res, y_res),
      (x_res + w_res, y_res + h_res),
      (0, 255, 0),
      2
  )

  # Centroides das duas faixas analisadas
  faixas_debug = [0,1,2]

  for idx in faixas_debug:

      M = cv2.moments(faixas[idx])

      px = int(M["m10"] / M["m00"])
      py = int(M["m01"] / M["m00"])

      # Centroide
      cv2.circle(imgRotacDebug, (px, py), 5, (0, 0, 255), -1)
      cv2.circle(imgRotacDebug2, (px, py), 5, (0, 0, 255), -1)


      # Distância até a borda esquerda (d1)
      cv2.line(
          imgRotacDebug,
          (x_res, py),
          (px, py),
          (255, 0, 0),
          2
         )

      # Distância até a borda direita (d2)
      cv2.line(
          imgRotacDebug2,
          (px, py),
          (x_res + w_res, py),
          (0, 255, 255),
          2
        )
      margem = 30

      x1 = max(0, x_res - margem)
      y1 = max(0, y_res - margem)

      x2 = min(imgRotacDebug.shape[1], x_res + w_res + margem)
      y2 = min(imgRotacDebug.shape[0], y_res + h_res + margem)

  imgRotacDebug  = imgRotacDebug[y1:y2, x1:x2]
  imgRotacDebug2 = imgRotacDebug2[y1:y2, x1:x2]

  cv2_imshow(imgRotacDebug)  #Desenha o retângulo, os centroides e a linha entre o centroide e a borda do retângulo (representando d_esq)
  cv2_imshow(imgRotacDebug2) #Desenha o retângulo, os centroides e a linha entre o centroide e a borda do retângulo (representando d_dir)

# **Decisor das cores**


In [ ]:
colors = []


for Faixa in FaixasOrdenadas:

    M = cv2.moments(Faixa)
    px = int(M["m10"] / M["m00"])
    py = int(M["m01"] / M["m00"])

    janela = blur[py-2:py+3, px-2:px+3]

    # Converte toda a janela para HSV
    janela_hsv = cv2.cvtColor(janela, cv2.COLOR_BGR2HSV)


    # Média dos pixels HSV da janela
    media_hsv = np.mean(janela_hsv, axis=(0, 1))
    h, s, v = media_hsv

    colors.append(np.array([
        h,
        s,
        v
    ], dtype=np.float32))

In [ ]:
if Debug == True:
    imgRotacDebug = imgRotac.copy()
    cv2.rectangle(
      imgRotacDebug,
      (px - 2, py - 2),
      (px + 3, py + 3),
      (0, 255, 0),
      1
    )
    imgRotacDebug  = imgRotacDebug[py -30:py +30, px -30 :px+30]
    cv2_imshow(imgRotacDebug) # Zoom na região da ultima faixa e desenha a janela retirada dessa faixa para decisão da cor
                              # Escolheu-se por desenhar somente a ultima faixa para não criar uma nova variável em meio ao código apenas para Debug
                              # O mesmo é feito para outras faixas


In [ ]:
corFaixas = []

for cor in colors:
  corFaixas.append(colorDec(cor))



## **Tratamento especial para mais de uma faixa preta**


In [ ]:
if corFaixas.count('Preto') > 1:
    minBlack = min(
        cor[2]
        for nome, cor in zip(corFaixas, colors)
        if nome == 'Preto'
    )

    for idxCor, cor in enumerate(colors):
        if corFaixas[idxCor] == 'Preto' and np.abs(cor[2] - minBlack) >= 5:
            corFaixas[idxCor] = colorDec(cor, canBlack=False)

In [ ]:
resValue = resistencia(corFaixas)

# **Ajustes para saída do programa**




In [ ]:
# Ajusta o resistor na orientação de leitura das Faixas para
# apresentar na saída do programa.

imgFinal = imgRotac2.copy()
FaixasOrdenadasFlip = [c.copy() for c in FaixasOrdenadas]
primeira = ordem[0]

if d_dir[primeira] < d_esq[primeira]:
    imgFinal = cv2.flip(imgFinal, 1)
    resistorFlip  = cv2.flip(resistor, 1)

    ContResFlip, _ = cv2.findContours(
        resistorFlip, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )

    largura = imgFinal.shape[1]


    for c in FaixasOrdenadasFlip:
        c[:, 0, 0] = largura - 1 - c[:, 0, 0]

    x_res, y_res, w_res, h_res = cv2.boundingRect(
      max(ContResFlip, key=cv2.contourArea)
        )



In [ ]:
for i, faixa in enumerate(FaixasOrdenadasFlip):

    x, y, w, h = cv2.boundingRect(faixa)

    cv2.rectangle(imgFinal, (x,y), (x+w,y+h), (255,0,0), 2)

    cv2.putText(
        imgFinal,
        str(i+1),
        (x, y-10),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255,0,0),
        2
    )



cx = x_res + w_res // 2
cy = y_res + h_res // 2

w_zoom = 2 * w_res
h_zoom = 4 * h_res

x1 = max(0, cx - w_zoom // 2)
y1 = max(0, cy - h_zoom // 2)

x2 = min(imgFinal.shape[1], cx + w_zoom // 2)
y2 = min(imgFinal.shape[0], cy + h_zoom // 2)

zoom = imgFinal[y1:y2, x1:x2].copy()

In [ ]:
if Debug == True:
  cv2_imshow(zoom) # Imagem com zoom em torno do resistor e com as BoundigBoxes dos contornos desenhadas e enumeradas.

In [ ]:


img_pil = Image.fromarray(cv2.cvtColor(zoom, cv2.COLOR_BGR2RGB))

draw = ImageDraw.Draw(img_pil)
font = ImageFont.truetype(
    "/usr/share/fonts/truetype/liberation2/LiberationSans-Regular.ttf", #Essa fonte funciona apenas no Colab
    32
)

draw.text((20, 20), f'Resistor de {resValue}', font=font, fill=(0,0,0))


texto = f'Resistor de {resValue}'

bbox = draw.textbbox((20, 20), texto, font=font)

draw.rectangle(
    bbox,
    fill="white"
)

draw.text(
    (20, 20),
    texto,
    font=font,
    fill="black"
)

zoom = cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)

cv2_imshow(zoom)
